## Side-by-side timeseries animations for Land Cover 2.0 and Geomedian

This notebook uses the [Land Cover 3.0](https://knowledge.dea.ga.gov.au/data/product/dea-land-cover-landsat/) and [Geometric Median and Median Absolute Deviation (Landsat)](https://knowledge.dea.ga.gov.au/data/product/dea-geometric-median-and-median-absolute-deviation-landsat/) products to create animations showing areas of interest throughout time in yearly timesteps.

This notebook connects to the development sandbox database currently; once Land Cover 3.0 is publicly accessible this can be updated and access to the database replaced with use of the `datacube` python package for accessing the datasets.

The notebook has a cell for the user to input parameters. These will then be passed to functions that will connect to the database, generate individual animations, and then generate paired animations. These will be saved out to the directory specified in the parameter cell.


In [1]:
%matplotlib inline

import os
import sys
import json
import pandas as pd
import numpy as np
import datacube
import geopandas as gpd
import rasterio.features
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patheffects as PathEffects
from datacube.utils.masking import make_mask
from matplotlib import colors as mcolours
from IPython.display import Image
from IPython.core.display import Video
from shapely.geometry import shape, box
from skimage.exposure import rescale_intensity
from pathlib import Path

from PIL import ImageSequence
from PIL import Image as pimg

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import xr_animation
from dea_tools.spatial import xr_vectorize, add_geobox
from dea_tools.landcover import lc_animation, lc_colourmap

The cell below is only required if trying to access data that is only in the development database. If you do not need to do that, you can comment out the cell and proceed using `datacube`

In [2]:

dc = datacube.Datacube(app="landcover_geomad_aminations")

In [3]:
# # If you do not need to access the dev sandbox use the following instead:

# dc = datacube.Datacube(app="landcover_geomad_aminations")

Place parameters in the cell below. Input parameters that change for each target location are placed in lists.

- `output_dir`: the directory where the individual animations will be saved
- `text_size`: size of the 'year' text
- `dpi`
- `lc_product`: the land cover product
- `gmad_product`: the geoMAD product used. In this case, the Landsat 7 GeoMAD.
- `buffers`: list of buffers used with central lat/lon coordinates to generate region of interest
- `intervals`: Control the speed of the animations by changing the itnerval between each frame.
- `lat_list`: list of the central latitude for each area of interest
- `lon_list`: list of the central longitude for each area of interest
- `time_list`: a list of tuples, each containing the start and end year for the desired timeframe
- `roi_name_list`: list containing a string name that will be appended to the output filenames

In [4]:
"""
Use the boolean flags to turn on/off sections of the notebook. If you only want to create the masked animations for example, turn core_animations and lonterm_animations to False.
This speeds up the notebook.

To run this notebook, you should supply a csv with each area you are interested stored as a row.

TODO: Include a template CSV with notebooks when work is all done.
"""
core_animations = True
masked_animations = True
longterm_animations = False

output_dir = 'output_gifs'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

text_size = 35
dpi = 150
lc_product = 'ga_ls_landcover_class_cyear_3'
gmad_product = ['ga_ls5t_gm_cyear_3', 'ga_ls7e_gm_cyear_3', 'ga_ls8cls9c_gm_cyear_3']

In [5]:
# load landcover colour library from json file
lc_colours_fpath = 'lc_colours.json'

with open(lc_colours_fpath, 'r') as file:
    colours_json = json.load(file)

In [6]:
# The '_slim' csv contains a minimal set of testing locations. Swap for a complete csv for final testing.

df = pd.read_csv('input_csv/roi_count_pixels_level3_4.csv')

lats = df['centre_y'].tolist()
lons = df['centre_x'].tolist()
buffers = [i/2 for i in df['buffer_size_wgs84']]
times = df.apply(lambda row: (str(row['start_year']), str(row['end_year'])), axis=1).tolist()
intervals = df['interval'].tolist()
roi_names = df['name'].tolist()

lc_levels = df['level'].tolist()


# Filter out rows where 'levle3_class' or 'level4_class' is -99, to only get rows where we want to create masked class animations
filtered_df = df[(df['level3_class'] != -99) | (df['level3_class'] != -99)]

# Create the lists from the filtered DataFrame
mask_lats = filtered_df['centre_y'].tolist()
mask_lons = filtered_df['centre_x'].tolist()
mask_buffers = filtered_df['buffer_size_wgs84'].tolist()
mask_times = filtered_df.apply(lambda row: (str(row['start_year']), str(row['end_year'])), axis=1).tolist()
mask_intervals = filtered_df['interval'].tolist()

mask_lc_levels = filtered_df['level'].tolist()
mask_lc_level3_classes = filtered_df['level3_class'].tolist()
mask_lc_level4_classes = filtered_df['level4_class'].apply(lambda x: [int(i) for i in x.split(',')])

mask_roi_names = filtered_df['name'].tolist()

# long term time series


In [7]:
mask_lc_level4_classes

2    [98, 99, 100, 101, 102, 103, 104]
5                                 [93]
Name: level4_class, dtype: object

In [8]:
def get_normalized_rgb(class_number):
    lc_colours_fpath = 'lc_colours.json'
    with open(lc_colours_fpath, 'r') as file:
        colours_json = json.load(file)
        
    colours = colours_json["lc_colours"]["level3"].get(str(class_number))
    if colours:
        return tuple(colour / 255 for colour in colours[:3])
    else:
        return None

In [9]:
def xr_animation_modified(ds,
                 bands=None,
                 output_path='animation.mp4',
                 width_pixels=500,
                 interval=100,
                 percentile_stretch=(0.02, 0.98),
                 image_proc_funcs=None,
                 show_gdf=None,
                 show_date='%d %b %Y',
                 show_text=None,
                 show_colorbar=True,
                 gdf_kwargs={},
                 annotation_kwargs={},
                 imshow_kwargs={},
                 colorbar_kwargs={},
                 limit=None,
                 list_extra_labels=[]):

    # this is a modified version of xr_animation that allows us to add text to animations. See original xr_animation function for more documentation.
    def _start_end_times(gdf, ds):
        # Make copy of gdf so we do not modify original data
        gdf = gdf.copy()

        # Get min and max times from input dataset
        minmax_times = pd.to_datetime(ds.time.isel(time=[0, -1]).values)

        # Update both `start_time` and `end_time` columns
        for time_col, time_val in zip(['start_time', 'end_time'], minmax_times):

            # Add time_col if it does not exist
            if time_col not in gdf:
                gdf[time_col] = np.nan

            # Convert values to datetimes and fill gaps with relevant time value
            gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
            gdf[time_col] = gdf[time_col].fillna(time_val)

        return gdf

    def _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults):
        # Create new axis object for colorbar
        cax = fig.add_axes([0.02, 0.02, 0.96, 0.03])

        # Initialise color bar using plot min and max values
        img = ax.imshow(np.array([[vmin, vmax]]), **imshow_defaults)
        fig.colorbar(img,
                     cax=cax,
                     orientation='horizontal',
                     ticks=np.linspace(vmin, vmax, 2))

        # Fine-tune appearance of colorbar
        cax.xaxis.set_ticks_position('top')
        cax.tick_params(axis='x', **colorbar_defaults)
        cax.get_xticklabels()[0].set_horizontalalignment('left')
        cax.get_xticklabels()[-1].set_horizontalalignment('right')

    def _frame_annotation(times, show_date, show_text):
        # Test if show_text is supplied as a list
        is_sequence = isinstance(show_text, (list, tuple, np.ndarray))

        # Raise exception if it is shorter than number of dates
        if is_sequence and (len(show_text) == 1):
            show_text, is_sequence = show_text[0], False
        elif is_sequence and (len(show_text) < len(times)):
            raise ValueError(f'Annotations supplied via `show_text` must have '
                             f'either a length of 1, or a length >= the number '
                             f'of timesteps in `ds` (n={len(times)})')

        times_list = (times.dt.strftime(show_date).values
                      if show_date else [None] * len(times))
        text_list = show_text if is_sequence else [show_text] * len(times)
        annotation_list = [
            '\n'.join([str(i)
                       for i in (a, b)
                       if i])
            for a, b in zip(times_list, text_list)
        ]

        return annotation_list

    def _update_frames(i, ax, extent, annotation_text, gdf, gdf_defaults,
                       annotation_defaults, imshow_defaults):

        # Clear previous frame to optimise render speed and plot imagery
        ax.clear()
        ax.imshow(array[i, ...].clip(0.0, 1.0),
                  extent=extent,
                  vmin=0.0,
                  vmax=1.0,
                  **imshow_defaults)

        # Add annotation text
        ax.annotate(annotation_text[i], **annotation_defaults)

        # Add geodataframe annotation
        if show_gdf is not None:

            # Obtain start and end times to filter geodataframe features
            time_i = ds.time.isel(time=i).values

            # Subset geodataframe using start and end dates
            gdf_subset = show_gdf.loc[(show_gdf.start_time <= time_i) &
                                      (show_gdf.end_time >= time_i)]

            if len(gdf_subset.index) > 0:

                # Set color to geodataframe field if supplied
                if ('color' in gdf_subset) and ('color' not in gdf_kwargs):
                    gdf_defaults.update({'color': gdf_subset['color'].tolist()})

                gdf_subset.plot(ax=ax, **gdf_defaults)

        # Remove axes to show imagery only
        ax.axis('off')


    # Add GeoBox and odc.* accessor to array using `odc-geo`
    try:
        ds = add_geobox(ds)
    except ValueError:
        raise ValueError("Unable to determine `ds`'s coordinate "
                         "reference system (CRS). Please assign a CRS "
                         "to the array before passing it to this "
                         "function, e.g.: "
                         "`ds.odc.assign_crs(crs='EPSG:3577')`")
    
    # Test if bands have been supplied, or convert to list to allow
    # iteration if a single band is provided as a string
    if bands is None:
        raise ValueError(f'Please use the `bands` parameter to supply '
                         f'a list of one or three bands that exist as '
                         f'variables in `ds`, e.g. {list(ds.data_vars)}')
    elif isinstance(bands, str):
        bands = [bands]

    # Test if bands exist in dataset
    missing_bands = [b for b in bands if b not in ds.data_vars]
    if missing_bands:
        raise ValueError(f'Band(s) {missing_bands} do not exist as '
                         f'variables in `ds` {list(ds.data_vars)}')

    # Test if time dimension exists in dataset
    if 'time' not in ds.dims:
        raise ValueError(f"`ds` does not contain a 'time' dimension "
                         f"required for generating an animation")

    # Set default parameters
    outline = [PathEffects.withStroke(linewidth=2.5, foreground='black')]
    annotation_defaults = {
        'xy': (1, 1),
        'xycoords': 'axes fraction',
        'xytext': (-5, -5),
        'textcoords': 'offset points',
        'horizontalalignment': 'right',
        'verticalalignment': 'top',
        'fontsize': 20,
        'color': 'white',
        'path_effects': outline
    }
    imshow_defaults = {'cmap': 'magma', 'interpolation': 'nearest'}
    colorbar_defaults = {'colors': 'white', 'labelsize': 12, 'length': 0}
    gdf_defaults = {'linewidth': 1.5}

    # Update defaults with kwargs
    annotation_defaults.update(annotation_kwargs)
    imshow_defaults.update(imshow_kwargs)
    colorbar_defaults.update(colorbar_kwargs)
    gdf_defaults.update(gdf_kwargs)

    # Get info on dataset dimensions
    height, width = ds.odc.geobox.shape
    scale = width_pixels / width
    left, bottom, right, top = ds.odc.geobox.extent.boundingbox

    # Prepare annotations
    annotation_list = _frame_annotation(ds.time, show_date, show_text)

    if len(list_extra_labels)>0: # if a list of extra labels is provided
        annotation_list = [f'{a}\n{b}' for a,b in zip(annotation_list, list_extra_labels)]

    # Prepare geodataframe
    if show_gdf is not None:
        show_gdf = show_gdf.to_crs(ds.odc.geobox.crs)
        show_gdf = gpd.clip(show_gdf, mask=box(
            left, bottom, right, top)).reindex(show_gdf.index).dropna(how='all')
        show_gdf = _start_end_times(show_gdf, ds)

    # Convert data to 4D numpy array of shape [time, y, x, bands]
    ds = ds[bands].to_array().transpose(..., 'variable')[0:limit, ...]
    array = ds.astype(np.float32).values

    # Optionally apply image processing along axis 0 (e.g. to each timestep)
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} ({remaining_s:.1f} ' \
                   'seconds remaining at {rate_fmt}{postfix})'
    if image_proc_funcs:
        print('Applying custom image processing functions')
        for i, array_i in tqdm(enumerate(array),
                               total=len(ds.time),
                               leave=False,
                               bar_format=bar_format,
                               unit=' frames'):
            for func in image_proc_funcs:
                array_i = func(array_i)
            array[i, ...] = array_i

    # Clip to percentiles and rescale between 0.0 and 1.0 for plotting
    vmin, vmax = np.quantile(array[np.isfinite(array)], q=percentile_stretch)

    # Replace with vmin and vmax if present in `imshow_defaults`
    if 'vmin' in imshow_defaults:
        vmin = imshow_defaults.pop('vmin')
    if 'vmax' in imshow_defaults:
        vmax = imshow_defaults.pop('vmax')

    # Rescale between 0 and 1
    array = rescale_intensity(array,
                              in_range=(vmin, vmax),
                              out_range=(0.0, 1.0))
    array = np.squeeze(array)  # remove final axis if only one band

    # Set up figure
    fig, ax = plt.subplots()
    fig.set_size_inches(width * scale / 72, height * scale / 72, forward=True)
    fig.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

    # Optionally add colorbar
    if show_colorbar & (len(bands) == 1):
        _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults)

    # Animate
    print(f'Exporting animation to {output_path}')
    anim = FuncAnimation(
        fig=fig,
        func=_update_frames,
        fargs=(
            ax,  # axis to plot into
            [left, right, bottom, top],  # imshow extent
            annotation_list,  # list of text annotations
            show_gdf,  # geodataframe to plot over imagery
            gdf_defaults,  # any kwargs used to plot gdf
            annotation_defaults,  # kwargs for annotations
            imshow_defaults),  # kwargs for imshow
        frames=len(ds.time),
        interval=interval,
        repeat=False)



    # Export animation to file
    if Path(output_path).suffix == '.gif':
        anim.save(output_path, writer='pillow')
    else:
        anim.save(output_path, dpi=72)

In [10]:
def odc_connect(lat, lon, buffer, time):
    """
    Connects to the Open Data Cube (ODC) and loads datasets for land cover and geomad products within a specified geographic and temporal range.

    Parameters:
    lat (float): Latitude of the center point
    lon (float): Longitude of the center point
    buffer (float): Buffer distance around the center point to define the area of interest
    time (str): Time range for the data query in the format ('YYYY, YYYY')

    Returns:
    tuple: A tuple containing:
        - ds_lc (xarray.Dataset): Dataset containing land cover data
        - ds_gmad (xarray.Dataset): Dataset containing geomad data
    """
    lat_range = (lat - buffer, lat + buffer)
    lon_range = (lon - buffer, lon + buffer)
    
    query ={
        'x': lon_range,
        'y': lat_range,
        'time': time
    }
    ds_lc = dc.load(product=lc_product,
                measurements=['level3', 'level4'],
                **query)
    ds_gmad = dc.load(product=gmad_product,
                  measurements=['nbart_red', 'nbart_green', 'nbart_blue'],
                  dask_chunks = {'time':2, 'x':2048, 'y':2048},
                  **query)
    
    return ds_lc, ds_gmad

In [11]:
def generate_single_geomad_animations(gmad_ds, roi_name, interval, filename_base):
    """
    Generates GIF animations for land cover levels 3 and 4, and geomad data.

    Parameters:
    gmad_ds (xarray.Dataset): Dataset containing geomad data.
    roi_name (str): Name of the region of interest.
    interval (int): Interval between frames in the animation.

    Returns:
    tuple: A tuple containing:
        - file_name_gmad (str): File path of the generated geomad animation GIF.
    """
    file_name_gmad = f'{output_dir}/geomedian_{filename_base}_timeseries.gif'

    #generate geomad animations
    xr_animation(ds=gmad_ds,
                bands=['nbart_red', 'nbart_green','nbart_blue'],
                output_path=file_name_gmad,
                interval=interval,
                width_pixels=350,
                show_colorbar=False,
                show_date = False,
                percentile_stretch=(0.02, 0.98),
                annotation_kwargs= {'fontsize': 25})
    plt.close()

    return file_name_gmad


In [12]:
def generate_single_landcov_animations(ds, roi_name, interval, filename_base, lc_class=3):
    """
    Generates GIF animations for land cover levels 3 and 4, and geomad data.

    Parameters:
    ds (xarray.Dataset): Dataset containing land cover data.
    roi_name (str): Name of the region of interest.
    interval (int): Interval between frames in the animation.
    lc_class (int): level 3 or level 4 land cover collection 3

    Returns:
    tuple: A tuple containing:
        - file_name_landcover (str): File path of the generated level 3 land cover animation GIF.
    """

    #generate landcover animation, using the lc_class to determine if its level3 or level4
    if lc_class == 3:
        file_name_landcover = f'{output_dir}/landcover-level3_{filename_base}_timeseries.gif'
        lc_animation(ds.level3,
                    file_name=f'{output_dir}/landcover-level3_{filename_base}_timeseries',
                    colour_bar=False,
                    label_ax=False,
                    animation_interval=interval,
                    width_pixels=7,
                    font_size=text_size,
                    dpi=dpi)
        
    elif lc_class == 4: 
        file_name_landcover = f'{output_dir}/landcover-level4_{filename_base}_timeseries.gif'
        lc_animation(ds.level4,
                    file_name=f'{output_dir}/landcover-level4_{filename_base}_timeseries',
                    colour_bar=False,
                    label_ax=False,
                    animation_interval=interval,
                    width_pixels=7,
                    font_size=text_size,
                    dpi=dpi)

    return file_name_landcover


In [13]:
def crop_to_aspect_ratio(frame, aspect_ratio):
    """
    Crop pixels from the left and bottom of the gif frames to ensure all animations have the same aspect ratios.
    """
    width, height = frame.size
    new_height = int(width / aspect_ratio)
    new_width = width

    # Calculate the amount to crop from the left
    left_crop = (width - new_width) // 2

    # Crop the frame
    return frame.crop((left_crop, 0, width, new_height))

In [14]:
def timeseries_animation(file_name_gmad, file_name_landcover, filename_base, aspect_ratio=None, crop=False):
    """
    Creates a side-by-side GIF animation combining geomad and land cover animations, ensuring they are synchronized.

    Parameters:
    file_name_gmad (str): File path of the geomad animation GIF.
    file_name_landcover (str): File path of the land cover animation GIF.
    aspect_ratio (float): The desired aspect ratio to crop to (width/height).

    Returns:
    str: File path of the combined animation GIF.
    """
    final_animations_dir = os.path.join(output_dir, 'final_animations')
    if not os.path.exists(final_animations_dir):
        os.makedirs(final_animations_dir)

    directory, filename = os.path.split(file_name_landcover)
    name, ext = os.path.splitext(filename)
    new_filename = f"geomedian-{name}{ext}"
    output_filepath = os.path.join(directory, 'final_animations', new_filename)

    gif_lc = pimg.open(file_name_landcover)
    gif_gmad = pimg.open(file_name_gmad)

    if crop is True:
        frames1 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_lc)]

    else:
        frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif_lc)]
        

    # Get dimensions of cropped gifs
    frame_width, frame_height = frames1[0].size

    # Set the figure size dynamically based on the size of the gifs
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(frame_width * 2 / dpi, frame_height / dpi))

    # Function to make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')

    # Adjust layout to minimize white space
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

    # Animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=interval)  # Adjust interval as needed

    animation_object.save(output_filepath, writer="Pillow")
    plt.close()

    return output_filepath

In [15]:
def timeseries_animation_three_frames(file_name_gmad, file_name_1, file_name_2, aspect_ratio=None, crop=False):
    """
    Creates a side-by-side GIF animation combining geomad and land cover animations, ensuring they are synchronized.

    Parameters:
    file_name_gmad (str): File path of the geomad animation GIF.
    file_name_landcover (str): File path of the land cover animation GIF.
    aspect_ratio (float): The desired aspect ratio to crop to (width/height).

    Returns:
    str: File path of the combined animation GIF.
    """

    directory, filename = os.path.split(file_name_1)
    name, ext = os.path.splitext(filename)
    
    new_filename = f"{name}_gmad_c2_c3{ext}"
    output_filepath = os.path.join(directory, 'final_animations', new_filename)

    gif_1 = pimg.open(file_name_1)
    gif_2 = pimg.open(file_name_2)
    gif_gmad = pimg.open(file_name_gmad)

    if crop is True:
        frames1 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_1)]
        frames3 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_2)]

    else:
        frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif_1)]
        frames3 = [frame.copy() for frame in ImageSequence.Iterator(gif_2)]
        

    # Get dimensions of cropped gifs
    frame_width, frame_height = frames1[0].size

    # Set the figure size dynamically based on the size of the gifs
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(frame_width * 3 / dpi, frame_height / dpi))

    # Function to make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax3.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax3.imshow(frames3[frame % len(frames3)])
        ax1.axis('off')
        ax2.axis('off')
        ax3.axis('off')

    # Adjust layout to minimize white space
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

    # Animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=interval)  # Adjust interval as needed

    animation_object.save(output_filepath, writer="Pillow")
    plt.close()

    return output_filepath

In [16]:
def generate_single_masked_animations(ds, gdf, filename_base, interval, class_number, prefix=None):
    class_colour = get_normalized_rgb(class_number)

    filename = f'{output_dir}/{prefix}_{filename_base}_timeseries.gif'
    
    xr_animation(ds=ds,
                bands=['nbart_red', 'nbart_green','nbart_blue'],
                output_path=filename,
                show_gdf=gdf,
                interval=interval,
                width_pixels=350,
                show_colorbar=False,
                show_date = '%Y',
                gdf_kwargs = {'color': class_colour},
                percentile_stretch=(0.02, 0.98),
                annotation_kwargs= {'fontsize': 25})
    plt.close()

    
    
    return filename


In [17]:
if core_animations is True:
    for lat, lon, buffer, time, interval, roi, lc_level in zip(lats, lons, buffers, times, intervals, roi_names, lc_levels):
        start_year = time[0]
        end_year = time[1]

        filename_base = f'{start_year}-{end_year}_{roi}' #this + filename info in functions to geerate consisten filenames
        print(filename_base)
        lc_ds, gm_ds = odc_connect(lat, lon, buffer, time)
        fname_gmad = generate_single_geomad_animations(gm_ds, roi, interval, filename_base)
        fname_lc = generate_single_landcov_animations(lc_ds, roi, interval, filename_base, lc_class=lc_level)
        timeseries_animation(fname_gmad, fname_lc, filename_base, aspect_ratio=None, crop=False)

2005-2015_VIC_bushfire
Exporting animation to output_gifs/geomedian_2005-2015_VIC_bushfire_timeseries.gif


  0%|          | 0/11 (0.0 seconds remaining at ? frames/s)

MovieWriter Pillow unavailable; using Pillow instead.


2000-2022_VIC_geomorphology
Exporting animation to output_gifs/geomedian_2000-2022_VIC_geomorphology_timeseries.gif


  0%|          | 0/23 (0.0 seconds remaining at ? frames/s)

...

  0%|          | 0/23 (0.0 seconds remaining at ? frames/s)

MovieWriter Pillow unavailable; using Pillow instead.


1990-2020_WA_urban_expansion
Exporting animation to output_gifs/geomedian_1990-2020_WA_urban_expansion_timeseries.gif


  0%|          | 0/31 (0.0 seconds remaining at ? frames/s)

MovieWriter Pillow unavailable; using Pillow instead.


### WARNING:

Currently the masked animations only work for level 3 land cover inputs. 

In [22]:
if masked_animations is True:
    for lat, lon, buffer, time, interval, roi_name, mask_lc_level, mask_lc_level3_class in zip(mask_lats, mask_lons, mask_buffers, mask_times, mask_intervals, mask_roi_names, mask_lc_levels, mask_lc_level3_classes):
            start_year=time[0]
            end_year=time[1]
            if mask_lc_level==3:
                gdfs = []
                class_code = mask_lc_level3_class
                prefix = 'landcover-level3-singleclass-geomedian'
            
                lat_range = (lat - buffer, lat + buffer)
                lon_range = (lon - buffer, lon + buffer)
                
                query ={
                    'x': lon_range,
                    'y': lat_range,
                    'time': time}
          
                ds_lc = dc.load(product=lc_product,
                        measurements=['level3'],
                        **query)
                ds_lc_lvl = ds_lc.level3

                mask_gm_ds = dc.load(product=gmad_product,
                                    measurements = ['nbart_blue', 'nbart_green', 'nbart_red'],
                                    dask_chunks = {'time':2, 'x':2048, 'y':2048},
                                    **query)
            
                for time in ds_lc_lvl.time:
                    start_year = str(time.data)[:4]
                    end_year = str(int(start_year)+1)
                    ds = ds_lc_lvl.sel(time=time)
                    
                    gdf = xr_vectorize(da = ds, 
                                       #mask = ds.values==class_code,
                                       mask = np.isin(ds.values, class_code),
                                       attribute_col='landcover_class_code')
                   
                    # take individual gdf's and combine them, including the year as an attribute. Needed for xr_animation
                    gdf['start_time']=start_year
                    gdf['end_time']=end_year
                    gdfs.append(gdf)
            
                combined_gdfs = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
                    #combined_gdfs.crs = gdfs[0].crs
                    #combined_gdfs.to_file(f'vector_classes/{roi_name}_lc_lvl3_combined_years.geojson', driver='GeoJSON')

        
                filename_base = f'{start_year}-{end_year}_{roi_name}' #this + filename info in functions to geerate consisten filenames
                mask_lvl3_fname = generate_single_masked_animations(mask_gm_ds, combined_gdfs, filename_base, interval, mask_lc_level3_class, prefix=prefix)
            else:
                print("Masked animations with level 4 landcover classes are not developed yet. ")


/env/lib/python3.10/site-packages/dea_tools/plotting.py:502: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
/env/lib/python3.10/site-packages/dea_tools/plotting.py:502: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')


Exporting animation to output_gifs/landcover-level3-singleclass-geomedian_2020-2021_NSW_lake_keepit_change_timeseries.gif


  0%|          | 0/31 (0.0 seconds remaining at ? frames/s)

Masked animations with level 4 landcover classes are not developed yet. 


In [ ]:
# if longterm_animations is True:
#     for lat, lon, buffer, time, interval, roi in zip(lt_lats, lt_lons, lt_buffers, lt_times, lt_intervals, lt_roi_names):
#         lc_ds, gm_ds = odc_connect(lat, lon, buffer, time)
#         fname_gmad = generate_single_geomad_animations(gm_ds, roi, interval)
#         fname_lc_lvl3 = generate_single_landcov_animations(lc_ds, roi, interval, lc_class=3)

#         timeseries_animation_three_frames(fname_gmad, fname_3, fname_4, aspect_ratio, crop=True)